# Constrained Variables

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating every continuous latent variable with a Gaussian. Many parameters are not free to take any real value: a probability lies between $0$ and $1$, a rate or a precision is positive. A Gaussian on such a variable puts mass where the model is undefined. The standard remedy is to approximate the variable with a Gaussian in an unconstrained space, the logit of a probability or the logarithm of a rate. The *Gaussian Mixture without Conjugacy* example of this section does this by hand, writing the model in terms of log-precisions.

`fallback = LocalLaplace()` does it automatically, without rewriting the model. The model keeps its natural priors, a Beta prior for a probability and a Gamma prior for a rate. The projection reads the domain of each variable from the support of its prior, computes the Laplace approximation in the unconstrained coordinates and returns the corresponding distribution on the original domain: a logit-normal for a probability, a log-normal for a rate.

The model is the zero-inflated Poisson regression from the "drunk monks" example of McElreath's *Statistical Rethinking* (2nd edition, Section 12.2). A monastery copies manuscripts at an average rate of $\lambda$ per day. On some days, with probability $p$, the monks drink instead and copy nothing. We observe only the daily number of manuscripts. A zero can come from a day of drinking or from a working day without a finished manuscript, so the two parameters are not separately visible in the data:

$$\begin{aligned}
p &\sim \mathrm{Beta}(1, 1),\\
\lambda &\sim \mathrm{Gamma}(2, 1),\\
y_i &\sim p\,\delta_0 + (1 - p)\,\mathrm{Poisson}(\lambda).
\end{aligned}$$

No exact message passing rule exists for the likelihood, and none is written in this notebook either.

In [ ]:
using RxInfer, BayesBase, Distributions, LinearAlgebra
using SpecialFunctions, StatsFuns, StableRNGs, Plots, PrettyTables, ExponentialFamilyProjection
import ExpectationApproximations as EA

## Laplace in transformed coordinates

Let $x = g(u)$ be a smooth bijection from the real line onto the domain of $x$. If $x$ has the density $p(x)$, then $u = g^{-1}(x)$ has the density

$$p_u(u) = p\left(g(u)\right)\,\left|g'(u)\right|.$$

$p_u$ lives on the whole real line, so its Laplace approximation $\mathcal{N}(\hat{u}, \sigma^2)$ is well defined, and pushing it forward through $g$ gives an approximation of $p(x)$ that respects the domain. For the domains of this example:

| Domain | $g(u)$ | $\log \lvert g'(u) \rvert$ | Pushforward of $\mathcal{N}(\hat{u}, \sigma^2)$ |
|---|---|---|---|
| $(-\infty, \infty)$ | $u$ | $0$ | $\mathcal{N}(\hat{u}, \sigma^2)$ |
| $(0, \infty)$ | $e^u$ | $u$ | log-normal |
| $(0, 1)$ | $1 / (1 + e^{-u})$ | $-\log(1 + e^{-u}) - \log(1 + e^{u})$ | logit-normal |

The domain is read from the `support` of the first factor in the product that has one, which is the prior. The pushforward is an `ExpectationApproximations.TransformedGaussian`, which keeps the Gaussian in unconstrained coordinates and has a `logpdf`, samples, a mean and a variance; expectations over it are computed through its Gaussian.

## A variational rule fallback

The two parameters share every likelihood factor, so we use a mean-field scheme, $q(p, \lambda) = q(p)\,q(\lambda)$. The message from a factor $f$ to $p$ is then $\exp\left(\mathbb{E}_{q(\lambda)}\left[\log f(y_i, p, \lambda)\right]\right)$, and vice versa. The fallback computes these expectations with the backends of the loaded packages: Gauss–Hermite quadrature through the Gaussian of a transformed belief, and, in the first iteration, when the beliefs are the Beta and Gamma priors themselves, Gauss–Jacobi and Gauss–Laguerre rules made for them.

## The likelihood node

The zero-inflated Poisson is a node of its own with the interfaces `out`, `p` and `λ`.

In [ ]:
struct ZeroInflatedPoisson{P, L} <: DiscreteUnivariateDistribution
    p::P
    λ::L
end

function BayesBase.logpdf(d::ZeroInflatedPoisson, y)
    y == 0 && return log(d.p + (1 - d.p) * exp(-d.λ))
    return log1p(-d.p) + y * log(d.λ) - d.λ - loggamma(y + 1)
end
BayesBase.insupport(::ZeroInflatedPoisson, y) = true

@define_factor_node(node = ZeroInflatedPoisson, type = Stochastic, interfaces = [:out, :p, :λ])

## Data

We follow McElreath's simulation: a year of records, monks who drink on one day in five, and one manuscript per working day on average.

In [ ]:
rng = StableRNG(2024)
N, p_true, λ_true = 365, 0.2, 1.0
drink = rand(rng, N) .< p_true
y = [d ? 0 : rand(rng, Poisson(λ_true)) for d in drink]

println("days with zero manuscripts: ", count(==(0), y), " of ", N, ", of them drinking days: ", count(drink))
histogram(y; bins = -0.5:1:(maximum(y) + 0.5), color = :gray, alpha = 0.6, label = "manuscripts per day", xlabel = "y")

## Inference

The model is written exactly as the mathematics above. The only hint about the approximation is the keyword; mean-field inference starts from the priors.

In [ ]:
@model function monastery(y, prior_p, prior_λ)
    p ~ prior_p
    λ ~ prior_λ
    for i in eachindex(y)
        y[i] ~ ZeroInflatedPoisson(p, λ)
    end
end

prior_p, prior_λ = Beta(1.0, 1.0), Gamma(2.0, 1.0)

mean_field = @constraints begin
    q(p, λ) = q(p)q(λ)
end

n_iterations = 20

result = infer(
    model = monastery(prior_p = prior_p, prior_λ = prior_λ),
    data = (y = y,),
    constraints = mean_field,
    iterations = n_iterations,
    returnvars = KeepEach(),
    free_energy = true,
    fallback = LocalLaplace(),
    report = true,
)

q_p, q_λ = result.posteriors[:p][end], result.posteriors[:λ][end]

The posteriors are a logit-normal for $p$ and a log-normal for $\lambda$, chosen by the supports of the Beta and Gamma priors. The free energy shows the convergence.

In [ ]:
plot(1:n_iterations, result.free_energy; xlabel = "iteration", ylabel = "free energy", label = "mean-field Laplace", marker = :circle)
@assert EA.domain(q_p) isa EA.UnitInterval && EA.domain(q_λ) isa EA.Positive # hide
@assert abs(result.free_energy[end] - result.free_energy[end - 5]) < 1e-6 * abs(result.free_energy[end]) # hide

## What was approximated

`report = true` recorded what the keyword did. Both posteriors are Laplace approximations in transformed coordinates, from the priors' exact messages and the likelihood's variational messages. Each likelihood message is an expectation over the other variable's belief, a logit-normal or a log-normal. The free energy's average energies of the likelihood come from the rule fallback too.

In [ ]:
@assert isempty(RxInfer.diagnostics(result.report)) # hide
@assert all(name -> result.report.approximations.variables[(name, nothing)].marginal.outcome.converged, (:p, :λ)) # hide
@assert RxInfer.node_types(result.report.approximations)[(ZeroInflatedPoisson, :energy, nothing)].fallbacks > 0 # hide
result.report.approximations

## References

1. **The exact posterior.** With two parameters, it can be evaluated on a fine grid.
2. **The joint transformed Laplace approximation.** A single Gaussian over $(\operatorname{logit} p, \log \lambda)$, the Laplace approximation of the joint posterior in transformed coordinates, computed directly with `laplace` from ExpectationApproximations.jl. It differs from our message passing result only in that it keeps the correlation between $p$ and $\lambda$, which the mean-field factorisation discards.

In [ ]:
logposterior(y, p, λ) = logpdf(prior_p, p) + logpdf(prior_λ, λ) + sum(logpdf(ZeroInflatedPoisson(p, λ), yi) for yi in y)

function grid_posterior(y; ps = range(0.0005, 0.9995; length = 1000), λs = range(0.005, 4; length = 800))
    L = [logposterior(y, p, λ) for p in ps, λ in λs]
    P = exp.(L .- maximum(L))
    P ./= sum(P)
    Pp, Pλ = vec(sum(P; dims = 2)), vec(sum(P; dims = 1))
    mp, mλ = sum(Pp .* ps), sum(Pλ .* λs)
    sp, sλ = sqrt(sum(Pp .* (ps .- mp) .^ 2)), sqrt(sum(Pλ .* (λs .- mλ) .^ 2))
    correlation = sum(P .* ((ps .- mp) * (λs .- mλ)')) / (sp * sλ)
    # marginal densities: probabilities divided by the grid step
    return (ps = ps, λs = λs, density_p = Pp ./ step(ps), density_λ = Pλ ./ step(λs), p = (mp, sp), λ = (mλ, sλ), correlation = correlation)
end

grid = grid_posterior(y)

function joint_laplace(y)
    D_p, D_λ = EA.UnitInterval(), EA.Positive()
    logdensity(u) = logposterior(y, EA.forward(D_p, u[1]), EA.forward(D_λ, u[2])) + EA.logjacobian(D_p, u[1]) + EA.logjacobian(D_λ, u[2])
    g = EA.laplace(logdensity, [EA.inverse(D_p, mean(prior_p)), EA.inverse(D_λ, mean(prior_λ))]).distribution
    return EA.pushforward(D_p, Normal(mean(g)[1], sqrt(cov(g)[1, 1]))), EA.pushforward(D_λ, Normal(mean(g)[2], sqrt(cov(g)[2, 2])))
end

joint_p, joint_λ = joint_laplace(y)

In [ ]:
moments(q) = (mean(q), std(q))
show_moments(label, (m, s)) = string(rpad(label, 22), rpad(round(m; digits = 3), 8), round(s; digits = 3))

println(rpad("", 22), "p: mean  sd      λ: mean  sd")
for (label, a, b) in [
    ("exact (grid)", grid.p, grid.λ),
    ("joint Laplace", moments(joint_p), moments(joint_λ)),
    ("mean-field Laplace", moments(q_p), moments(q_λ)),
]
    println(show_moments(label, a), "    ", show_moments("", b)[23:end])
end
println("posterior correlation of p and λ (grid): ", round(grid.correlation; digits = 3))
@assert mean(q_p) > grid.p[1] && mean(joint_p) > grid.p[1] # hide
@assert std(q_p) < 0.8 * grid.p[2] && std(q_λ) < 0.8 * grid.λ[2] # hide
@assert abs(std(joint_p) - grid.p[2]) < 0.15 * grid.p[2] && abs(std(joint_λ) - grid.λ[2]) < 0.15 * grid.λ[2] # hide
@assert grid.correlation > 0.5 # hide

In [ ]:
plt_p = plot(grid.ps, grid.density_p; color = :black, label = "exact (grid)", xlabel = "p", xlims = (0, 0.5))
plot!(plt_p, grid.ps, pdf.(joint_p, grid.ps); color = :blue, linestyle = :dot, label = "joint Laplace")
plot!(plt_p, grid.ps, pdf.(q_p, grid.ps); color = :red, linestyle = :dash, label = "mean-field Laplace")

plt_λ = plot(grid.λs, grid.density_λ; color = :black, label = "exact (grid)", xlabel = "λ", xlims = (0.6, 1.6))
plot!(plt_λ, grid.λs, pdf.(joint_λ, grid.λs); color = :blue, linestyle = :dot, label = "joint Laplace")
plot!(plt_λ, grid.λs, pdf.(q_λ, grid.λs); color = :red, linestyle = :dash, label = "mean-field Laplace")

plot(plt_p, plt_λ; layout = (1, 2), size = (900, 350), bottom_margin = 5Plots.mm)

The two approximations make two different errors, and the comparison separates them:

- **Location.** Both Laplace approximations report a larger $p$ than the exact posterior. The posterior of $p$ is skewed, and a Laplace approximation is centred at the mode in transformed coordinates, not at the mean. This is the error of the Laplace approximation itself, which looks only at the mode; an approximation that weighs the whole density, such as natural-gradient propagation's, `fallback = NaturalGradientPropagation()`, is not tied to it.
- **Width.** The joint approximation has nearly the exact standard deviations, the mean-field approximation is about a third too narrow. Zero-inflated models are a hard case for mean field: a zero is explained either by a drinking day or by a working day without a manuscript, so $p$ and $\lambda$ are strongly correlated a posteriori, and a factorised approximation is overconfident. This is the error of the factorisation, and what a joint Gaussian over a block of variables would correct.

## Sanity check: a conjugate model

Without zero inflation, the Gamma prior is conjugate to the Poisson likelihood and the posterior of $\lambda$ is $\mathrm{Gamma}(a, b)$ in the shape-rate parametrisation. We check two things.

1. **The keyword does not touch exact rules.** RxInfer has conjugate rules for `Poisson`, so with `y .~ Poisson(λ)` the fallback never runs, and the posterior is the exact Gamma. Its report lists no approximated posterior.
2. **The projection is the transformed Laplace approximation.** If the likelihood is a node of our own, `PoissonRate`, without rules, the fallback runs, and the posterior must be the transformed Laplace approximation computed by hand. In log coordinates, $u = \log \lambda$, the log-density of $\mathrm{Gamma}(a, b)$ is $a u - b e^{u}$ up to a constant, with the mode $\hat{u} = \log(a / b)$ and the curvature $-a$ there, so the approximation is $\mathrm{LogNormal}(\log(a/b), 1/\sqrt{a})$.

In [ ]:
struct PoissonRate{L} <: DiscreteUnivariateDistribution
    λ::L
end

BayesBase.logpdf(d::PoissonRate, y) = y * log(d.λ) - d.λ - loggamma(y + 1)
BayesBase.insupport(::PoissonRate, y) = true

@define_factor_node(node = PoissonRate, type = Stochastic, interfaces = [:out, :λ])

@model function poisson_counts(y, prior_λ, likelihood)
    λ ~ prior_λ
    for i in eachindex(y)
        y[i] ~ likelihood(λ)
    end
end

counts = rand(StableRNG(11), Poisson(λ_true), 20)
a, b = shape(prior_λ) + sum(counts), 1 / scale(prior_λ) + length(counts)

exact_result = infer(
    model = poisson_counts(prior_λ = prior_λ, likelihood = Poisson),
    data = (y = counts,),
    fallback = LocalLaplace(),
    report = (:approximations,),
)
@assert all(record -> record.fallbacks == 0, values(RxInfer.node_types(exact_result.report.approximations))) # hide
fallback_result = infer(
    model = poisson_counts(prior_λ = prior_λ, likelihood = PoissonRate),
    data = (y = counts,),
    fallback = LocalLaplace(),
)

println("exact rules:  ", exact_result.posteriors[:λ], ",  exact: Gamma(shape = ", a, ", rate = ", b, ")")
println("fallback:     ", fallback_result.posteriors[:λ].gaussian, " in log coordinates,  by hand: ", Normal(log(a / b), 1 / sqrt(a)))
@assert mean(exact_result.posteriors[:λ]) ≈ a / b && var(exact_result.posteriors[:λ]) ≈ a / b^2 # hide
@assert isapprox(mean(fallback_result.posteriors[:λ].gaussian), log(a / b); rtol = 1e-8) && isapprox(std(fallback_result.posteriors[:λ].gaussian), 1 / sqrt(a); rtol = 1e-8) # hide

In [ ]:
exact_result.report

In [ ]:
exact_λ = Gamma(a, 1 / b)
println(rpad("", 22), "mean    sd")
println(show_moments("exact Gamma", moments(exact_λ)))
println(show_moments("transformed Laplace", moments(fallback_result.posteriors[:λ])))
println(show_moments("Gaussian Laplace", ((a - 1) / b, sqrt(a - 1) / b))) # mode and curvature of the Gamma in λ
@assert abs(mean(fallback_result.posteriors[:λ]) - mean(exact_λ)) < abs((a - 1) / b - mean(exact_λ)) # hide

The transformed Laplace approximation is closer to the exact posterior than a Gaussian Laplace approximation in $\lambda$ would be, because the log-normal is skewed in the right direction.

## Why the transform matters

What would happen with a Gaussian on the real line for $p$ and $\lambda$? The experiment behind this example ran exactly that, and it fails on the same year of data, for two reasons. The first is that the Newton search for the mode of $q(\lambda)$, started at the prior mean $\lambda = 2$, overshoots below zero, where the product of the messages cannot be evaluated. A more careful line search would fix that. The second reason is fundamental. Under mean field, the message to $\lambda$ is an expectation over $q(p)$, and a Gaussian $q(p)$ has tails below $0$. Quadrature, like any other method of computing that expectation, then evaluates the likelihood at negative probabilities. A Gaussian with the moments of our posterior of $p$ already puts Gauss–Hermite nodes there:

In [ ]:
nodes, _ = EA.gausshermite(16)
gaussian_nodes = mean(q_p) .+ sqrt(2) * std(q_p) .* nodes
println("Gauss–Hermite nodes for a Gaussian q(p) below zero: ", count(<(0), gaussian_nodes), " of 16")
@assert count(<(0), gaussian_nodes) > 0 # hide

The transformed approximation also behaves well when little data pushes the posterior against a boundary. Take a new monastery with two weeks of records, whose monks rarely drink. The posterior of $p$ then piles up at $p = 0$.

In [ ]:
rng_small = StableRNG(5)
y_small = [rand(rng_small) < 0.05 ? 0 : rand(rng_small, Poisson(λ_true)) for _ in 1:14]
println("data: ", y_small)

grid_small = grid_posterior(y_small; λs = range(0.005, 6; length = 800))
small_result = infer(
    model = monastery(prior_p = prior_p, prior_λ = prior_λ),
    data = (y = y_small,),
    constraints = mean_field,
    iterations = n_iterations,
    fallback = LocalLaplace(),
)
small_p = small_result.posteriors[:p][end]

println(rpad("", 22), "p: mean  sd")
println(show_moments("exact (grid)", grid_small.p))
println(show_moments("mean-field Laplace", moments(small_p)))
@assert 0 < mean(small_p) < 0.4 # hide
@assert argmax(grid_small.density_p) == 1 # hide

In [ ]:
plt_small = plot(grid_small.ps, grid_small.density_p; color = :black, label = "exact (grid)", xlabel = "p", ylabel = "density")
plot!(plt_small, grid_small.ps, pdf.(small_p, grid_small.ps); color = :red, linestyle = :dash, label = "mean-field Laplace")
plt_small

Inference runs, and the result respects the domain, but it is visibly a Laplace approximation. The exact density has its maximum at $p = 0$, while every logit-normal density vanishes there. The transform makes the approximation well defined, not exact.

## A Beta posterior

A Beta distribution fits a probability near a boundary better. ExponentialFamilyProjection projects a belief onto any exponential family: here `q(p)` onto a Beta, by its natural gradient. `ExpectationStrategy` computes the projection's expectations with ExpectationApproximations' quadrature, Gauss–Jacobi for the Beta, so the projection is deterministic. The constraint is the variable's own, so $\lambda$ keeps the default, its log-normal.

In [ ]:
beta_strategy = ExpectationStrategy(EA.EADefault())
beta_constraints = @constraints begin
    q(p, λ) = q(p)q(λ)
    q(p)::ProjectedTo(Beta; parameters = ProjectionParameters(strategy = beta_strategy, niterations = 1000, tolerance = 1e-10))
end

beta_result = infer(
    model = monastery(prior_p = prior_p, prior_λ = prior_λ),
    data = (y = y_small,),
    constraints = beta_constraints,
    iterations = n_iterations,
    fallback = LocalLaplace(),
    report = true,
)
beta_p = beta_result.posteriors[:p][end]

println(rpad("", 22), "p: mean  sd")
println(show_moments("exact (grid)", grid_small.p))
println(show_moments("mean-field Laplace", moments(small_p)))
println(show_moments("mean-field Beta", moments(beta_p)))
@assert beta_p isa Beta # hide
@assert abs(mean(beta_p) - grid_small.p[1]) < 0.02 < abs(mean(small_p) - grid_small.p[1]) # hide
@assert isempty(RxInfer.diagnostics(beta_result.report)) # hide

In [ ]:
plot!(plt_small, grid_small.ps, pdf.(beta_p, grid_small.ps); color = :blue, linestyle = :dot, label = "mean-field Beta")

The Beta's mean is close to the exact one, where the logit-normal's is far off, and its density is high near $p = 0$, where the exact one is highest. The report shows the projection with its evidence lower bound:

In [ ]:
beta_result.report.approximations

## Exact vs approximate, and what is still missing

- **What worked.** The model was written with its natural Beta and Gamma priors and was not rewritten in unconstrained coordinates, and it needed no initialisation. The projection chose the logit and log transforms from the supports of the priors and returned logit-normal and log-normal posteriors, which converge in a few iterations. On the conjugate Poisson-Gamma model the keyword keeps the exact rules; through a node without rules, the result equals the transformed Laplace approximation computed by hand, which is closer to the exact posterior than a Gaussian one.
- **Why the transform.** Without it, inference does not run at all, even with a year of data: a Gaussian marginal of a probability has tails below zero, and the variational messages evaluate the likelihood there. With it, inference runs also when two weeks of data push the posterior against the boundary.
- **Accuracy.** The approximation is not exact, and the comparison with the grid and with the joint Laplace approximation shows where the error comes from. The means are those of the mode in transformed coordinates; the posterior is skewed and the means are off, an error of Laplace's method rather than of the transform. The standard deviations of the mean-field approximation are too small, because the posterior correlation of $p$ and $\lambda$ is strong; the joint approximation, a single Gaussian over $(\operatorname{logit} p, \log \lambda)$, gets them nearly right.
- **What is still missing.**
  - Message passing over a block of variables with different domains, here $(p, \lambda)$ as one variable, which would give the joint approximation.
  - A Beta for $p$ is one constraint away (above). Choosing the family by itself, from the prior, is not automatic yet.
  - The domain comes from the support of a prior. A constrained variable without such a factor, for example the output of a deterministic function, has no domain to read.